In [38]:
from torch.utils.data import Dataset
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import torch
from typing import Tuple
import os

In [6]:
class EgammaNpzDataset(Dataset):
    def __init__(self, file_paths, transform=True, percentage=1.0):
        self.file_paths: list[str] = file_paths
        self.index_last_ring = 101
        self.percentage: int = percentage
        self.percentage_dim: int = None

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        self.percentage_dim = None
        features, labels = None, None
        with np.load(self.file_paths[idx]) as samples:
            features, labels = self._create_rings(samples['data'],
                                                  samples['target'])
        features_tensor = torch.from_numpy(features).float()  
        labels_tensor = torch.from_numpy(labels).long()
        return features_tensor, labels_tensor, self.file_paths[idx]

    def _create_rings(self, data: np.ndarray,
                      target: np.ndarray) -> Tuple[Tuple[np.ndarray,np.ndarray],
                                                Tuple[np.ndarray,np.ndarray]]:
        
        signal_data = data[np.where(target == 1)]
        bg_data = data[np.where(target == 0)]

        signal_data = signal_data[:, 1: self.index_last_ring]
        bg_data = bg_data[:, 1: self.index_last_ring]
        indexes = get_rings_index(self.percentage)
        signal_data = signal_data[:, indexes]
        bg_data = bg_data[:, indexes]
        self.percentage_dim = len(indexes)

        dataset = np.concatenate(
            (
                norm1(signal_data),
                norm1(bg_data)
            ),
            axis=0
        )
        y_sinal = np.ones((signal_data.shape[0], 1), dtype=np.float32)
        y_background = np.zeros((bg_data.shape[0], 1), dtype=np.float32)
        y = np.vstack([y_sinal, y_background]).ravel()
        return dataset, y

    def _get_rings_index(self, percentage: float) -> list[int]:
        PreSampler_index = [index for index in range(0, 8)]
        EM1_index = [index for index in range(8, 72)]
        EM2_index = [index for index in range(72, 80)]
        EM3_index = [index for index in range(80, 88)]
        TileCal_index = [index for index in range(88, 100)]
        
        PreSampler_index_cap = round(len(PreSampler_index) * percentage)
        EM1_index_cap = round(len(EM1_index) * percentage)
        EM2_index_cap = round(len(EM2_index) * percentage)
        EM3_index_cap = round(len(EM3_index) * percentage)
        TileCal_index_cap = round(len(TileCal_index) * percentage)
    
        PreSampler_index = PreSampler_index[: PreSampler_index_cap]
        EM1_index = EM1_index[: EM1_index_cap]
        EM2_index = EM2_index[: EM2_index_cap]
        EM3_index = EM3_index[: EM3_index_cap]
        TileCal_index = TileCal_index[: TileCal_index_cap]
    
        return PreSampler_index + EM1_index + EM2_index + EM3_index + TileCal_index

    def _get_data_dim(self) -> int:
        if self.percentage_dim != None:
            return self.percentage_dim
        else:
            raise Exception("Dimension was not set.")

In [7]:
array1 = np.array([1, 2, 3, 4, 5])
array2 = np.random.rand(3, 3)
array3 = np.arange(10, 20, 2)

In [8]:
np.savez('my_data1.npz', data=np.array([1, 1, 1, 1, 1]), target=array2, sequence=array3)
np.savez('my_data2.npz', data=np.array([2,2,2,2,2]), target=array2, sequence=array3)
np.savez('my_data3.npz', data=np.array([3,3,3,3,3]), target=array2, sequence=array3)

In [9]:
file_path = ["my_data1.npz", "my_data2.npz", "my_data3.npz"]

In [41]:
from sklearn.model_selection import KFold
dataset= EgammaNpzDataset(file_path)

# kfold = KFold(3,shuffle=True)
# data = dataset[0]
# for index, file in enumerate(file_path):
#     print(f"{index = }")
#     for data_index_1, data_index_2 in kfold.split(dataset[index][0]):
#         data, target, path = dataset[index]
#         print(f"{data_index_1 = }")
#         print(f"{data[data_index_1] = }")
#         # print(f"{target = }")
#         # print(f"{path = }")
dataset[0]

(array([1, 1, 1, 1, 1]),
 array([[0.33083733, 0.54258162, 0.01855069],
        [0.28905703, 0.49242243, 0.31009042],
        [0.78794614, 0.60125826, 0.10135337]]),
 'my_data1.npz')

In [43]:
dataset0 = dataset[0]
dataset0[0]

array([1, 1, 1, 1, 1])

In [12]:
train_dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

In [62]:
for x in train_dataloader:
    print(x[0])

my_data1.npz
tensor([[1., 1., 1., 1., 1.]])
my_data2.npz
tensor([[2., 2., 2., 2., 2.]])
my_data3.npz
tensor([[3., 3., 3., 3., 3.]])


In [ ]:
class Trainer:
    def __init__(self, percentage : int = 1,
                 kernel_size : int = 4,
                 num_filters : int = 2,
                 k : int = 10,
                 n_repeats : int= 1,
                 ns = 100000, 
                 epochs = 25, 
                 batch_size = 256,
                 et_range = np.arange(0, 2), 
                 eta_range = np.arange(0, 1),
                 folder_path = None,
                 model_tag: str = "V1"
                ):
        self.all_training_results = []
        self.percentage = percentage
        self.epochs = epochs
        self.batch_size = batch_size
        self.ns = ns
        self.kernel_size = kernel_size
        self.num_filters = num_filters
        self.k = k
        self.n_repeats = n_repeats
        self.random_state = 42
        self.kf = StratifiedKFold(n_splits=self.k,
                   shuffle=True,
                   random_state=self.random_state)
        self.et_range = et_range
        self.eta_range = eta_range
        self.current_eta = -1
        self.current_et = -1
        self.folder_path = folder_path
        self.drive_path = '/eos/user/j/jlieberm/photonRinger/datasets/notIso'
        self.model_tag = model_tag
        self.index_last_ring = 101
        self.debug = False
        self.eGamma_dataset = EgammaNpzDataset()

    def train(self) -> Tuple[np.ndarray,np.ndarray]:
        self.all_training_results = []
        dataset, y = self.get_data(self.current_et, self.current_eta)
        for fold, (train_index, test_index) in enumerate(self.kf.split(dataset, y)):
            X_train, X_test = dataset[train_index], dataset[test_index]
            y_train, y_test = y[train_index], y[test_index]
            class_weights = compute_class_weight(class_weight="balanced",
                                                 classes=np.unique(y_train),
                                                 y=y_train)
            class_weights = dict(enumerate(class_weights))
            for repeat in range(self.n_repeats):
                print(f'\n--- Fold: {fold+1}/{self.k}, Inicialização: {repeat+1}/{self.n_repeats} ---')
                # Instancie o callback SP
                sp_callback = SP.sp(verbose=False, save_the_best=True, patience=10,
                                    kernel_size=self.kernel_size, 
                                    num_filters=self.num_filters)
                sp_callback.set_validation_data((X_test, y_test))
                # Construa um novo modelo para cada rodada para garantir pesos independentes
                model = build_model(self.percentage_dim, self.model_tag)
                # Treine o modelo
                # https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html
                history_object = model.fit(X_train, y_train,
                                          epochs = self.epochs,
                                          batch_size = self.batch_size,
                                          validation_data = (X_test, y_test),
                                          callbacks = [sp_callback],
                                          verbose = 0,
                                          class_weight = class_weights) # Definido como 0 para que apenas o callback imprima
                # Coletar informações relevantes após o treinamento desta rodada
                best_weights_for_this_run = sp_callback.get_best_model_weights()
                best_sp_for_this_run = sp_callback.get_best_sp_value()
                best_fa_for_this_run = sp_callback.get_best_fa_at_knee()
                best_pd_for_this_run = sp_callback.get_best_pd_at_knee()
                # Armazenar todas as informações para análise posterior
                if best_weights_for_this_run is not None:
                    self.all_training_results.append({
                        'fold': fold,
                        'repeat': repeat,
                        'best_sp_value': best_sp_for_this_run,
                        'best_fa_value': best_fa_for_this_run,
                        'best_pd_value': best_pd_for_this_run,
                        'best_weights': best_weights_for_this_run,
                        'keras_history': history_object.history # Objeto History do Keras para análise de métricas de época
                    })
                else:
                    print(f"Aviso: Nenhum peso foi salvo para Fold {fold+1}, Inicialização {repeat+1}. SP inicial pode ter sido 0.")
        print("\nTreinamento da validação cruzada concluído.")
        return dataset, y


In [44]:
def calculate_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch: y_true (shape {y_true.shape}) and "
                         f"y_pred (shape {y_pred.shape}) must have the same shape.")
    
    total_samples = len(y_true)
    
    if total_samples == 0:
        return 0.0  # Or np.nan, depending on desired behavior
        
    correct_predictions = np.sum(y_true == y_pred)
    
    return correct_predictions / total_samples

In [47]:
class Trainer:
    def __init__(self, sys_argv=None, n_splits=10,folder_path=None, 
                 batch_size=256, epochs=100,
                percentage=1, model_tag="V0",
                et_range = np.arange(0, 2), 
                 eta_range = np.arange(0, 1),
                debug=False):
        if sys_argv is not None:
            sys_argv = sys_argv[1:]

        self.num_workers = 2

        self.use_cuda = torch.cuda.is_available()
        self.device = torch.device("cuda" if self.use_cuda else "cpu")
        
        self.kfold = KFold(n_splits=n_splits, 
                           shuffle=True, 
                           random_state=42)
        self.folder_path = folder_path
        self.batch_size = batch_size
        self.epochs = epochs
        self.percentage = percentage
        self.model_tag = model_tag
        self.et_range = et_range
        self.eta_range = eta_range
        self.drive_path = '/eos/user/j/jlieberm/photonRinger/datasets/notIso'
        self.debug = debug
        
        self.all_y_preds_list = []
        self.all_y_true_list = []
        self.totalTrainingSamples_count = 0 

    def initModel(self):
        log.info("Initializing model")
        model = SuaClasseDeModelo() 

        if self.use_cuda:
            log.info(f"Using CUDA; {torch.cuda.device_count()} devices")
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model.to(self.device)
        return model

    def initOptimizer(self):
        return torch.optim.Adam(self.model.parameters(), lr=0.001)

    def initDataLoader(self, data, labels):
        features_tensor = torch.from_numpy(data).float()
        labels_tensor = torch.from_numpy(labels).float().view(-1, 1)

        dataset = TensorDataset(features_tensor, labels_tensor)
        batch_size = self.batch_size
        if self.use_cuda:
            batch_size *= torch.cuda.device_count()
            
        dataloader = DataLoader(dataset, batch_size=batch_size,
                                num_workers=self.num_workers,
                                pin_memory=self.use_cuda)
        return dataloader

    def main(self, index: int):
        data, target, path = self.full_dataset[index]
        print(f"{path = }")
        self.all_training_results = []
        for fold_idx, (train_index, val_index) in enumerate(self.kfold.split(data, target)):
            
            self.model = self.initModel()
            self.optimizer = self.initOptimizer()
            
            train_dl = self.initDataLoader(data[train_index],
                                           target[train_index])
            val_dl = self.initDataLoader(data[val_index],
                                           target[val_index])
            
            sp_tracker = SPCallbackPyTorch(self.model, patience=5, verbose=True)
            
            fold_history = {
                'train_loss': [], 'train_acc': [],
                'val_loss': [], 'val_acc': [],
                'val_sp': [], 'val_fa_at_knee': [], 'val_pd_at_knee': []
            }
            
            for epoch_ndx in range(1, self.epochs + 1):
                
                avg_train_loss, avg_train_acc = self.doTraining(epoch_ndx, train_dl)
                self.all_y_preds_list = [] 
                self.all_y_true_list = []
                avg_val_loss, avg_val_acc = self.doValidation(epoch_ndx, val_dl)
                
                stop_training, logs = sp_tracker.on_epoch_end(epoch_ndx, 
                                                              self.all_y_true_list, 
                                                              self.all_y_preds_list)
                
                fold_history['train_loss'].append(avg_train_loss)
                fold_history['train_acc'].append(avg_train_acc)
                fold_history['val_loss'].append(avg_val_loss)
                fold_history['val_acc'].append(avg_val_acc)
                fold_history['val_sp'].append(logs.get('max_sp_val', 0.0))
                fold_history['val_fa_at_knee'].append(logs.get('max_sp_fa_val', 0.0))
                fold_history['val_pd_at_knee'].append(logs.get('max_sp_pd_val', 0.0))

                if stop_training:
                    print(f"Fold {fold_idx}: Early stopping acionado na época {epoch_ndx}.")
                    break 
            
            
            best_weights_for_this_run = sp_tracker.get_best_model_weights()
            best_sp_for_this_run = sp_tracker.get_best_sp_value()
            best_fa_for_this_run = sp_tracker.get_best_fa_at_knee()
            best_pd_for_this_run = sp_tracker.get_best_pd_at_knee()
            
            if best_weights_for_this_run is not None:
                self.all_training_results.append({
                    'file_path': path, 
                    'fold': fold_idx,
                    'best_sp_value': best_sp_for_this_run,
                    'best_fa_value': best_fa_for_this_run,
                    'best_pd_value': best_pd_for_this_run,
                    'best_weights': best_weights_for_this_run,
                    'history': fold_history 
                })
        return path

    def doTraining(self, epoch_ndx: int, train_dl: DataLoader):
        log.info(f"Starting training epoch {epoch_ndx}...")
        self.model.train() 

        running_loss = 0.0
        running_corrects = 0
        total_samples = 0

        batch_iter = enumerate(train_dl)
        for batch_ndx, batch_tup in batch_iter:
            self.optimizer.zero_grad()
            
            loss_var, corrects_batch = self.computeBatchLoss(
                batch_ndx,
                batch_tup,
                train_dl.batch_size,
                validation_step=False 
            )
            
            loss_var.backward() 
            self.optimizer.step()
            
            batch_size = batch_tup[0].size(0)
            running_loss += loss_var.item() * batch_size
            running_corrects += corrects_batch
            total_samples += batch_size

        self.totalTrainingSamples_count += len(train_dl.dataset)
        
        epoch_loss = running_loss / total_samples
        epoch_acc = running_corrects / total_samples
        
        return epoch_loss, epoch_acc


    def doValidation(self, epoch_ndx: int, val_dl: DataLoader):
        log.info(f"Starting validation epoch {epoch_ndx}...")
        self.model.eval() 
        running_loss = 0.0
        running_corrects = 0
        total_samples = 0
        
        with torch.no_grad(): 
            for batch_ndx, batch_tup in enumerate(val_dl):
                
                loss_var, corrects_batch = self.computeBatchLoss(
                    batch_ndx,
                    batch_tup,
                    val_dl.batch_size,
                    validation_step=True 
                )
                
                batch_size = batch_tup[0].size(0)
                running_loss += loss_var.item() * batch_size
                running_corrects += corrects_batch
                total_samples += batch_size
        
        epoch_loss = running_loss / total_samples
        epoch_acc = running_corrects / total_samples
        
        return epoch_loss, epoch_acc


    def computeBatchLoss(self, batch_ndx: int, 
                         batch_tup: tuple[torch.Tensor, torch.Tensor],
                         batch_size: int,
                         validation_step: bool = False):
        
        input_t, label_t = batch_tup
        input_g = input_t.to(self.device, non_blocking=True)
        label_g = label_t.to(self.device, non_blocking=True)
        
        preds_prob_g = self.model(input_g)
        
        loss_func = nn.BCELoss()
        loss_g = loss_func(preds_prob_g, label_g) 

        preds_label_g = (preds_prob_g >= 0.5).float()
        corrects_batch = (preds_label_g == label_g).sum().item()

        if validation_step:
            self.all_y_preds_list.append(preds_prob_g.cpu().detach().numpy())
            self.all_y_true_list.append(label_g.cpu().detach().numpy())
            
        return loss_g, corrects_batch

    def _get_results_file_name(self, et: int, eta: int) -> str:
         return "iet{iet}.ieta{ieta}.pkl".format(ieta = eta,
                                                 iet = et)
        
    
    def save_results(self, folder_path: str, et: int, eta: int) -> None:
        all_training_results_template = self._get_results_file_name(et, eta)
        pd.DataFrame(self.all_training_results).to_pickle(
            os.path.join(folder_path, 
                         all_training_results_template))

    
    def verify_results(self, folder_path: str) -> bool:
        file2verify = os.path.join(folder_path, self._get_results_file_name())
        print(f"Verifying {file2verify}")
        if os.path.exists(file2verify):
            print(f"{file2verify} already processed")
            return False
        return True

    def run(self) -> None:
        if self.folder_path == None: 
            folderTemplateName = "model{model_tag}.dim{input_dim}.folds{folds}_id{id}".format(
                input_dim = self.percentage,
                model_tag = self.model_tag,
                folds=self.k, 
                id=datetime.now().strftime("%Y%m%d%H%M%S"))
            self.folder_path = str(create_folder(folderTemplateName))

        data_folder = [os.path.join(self.drive_path, file) for file in os.listdir(self.drive_path) if file.endswith(".npz")]
        self.full_dataset = EgammaNpzDataset(data_folder)

        for index, file in enumerate(data_folder):
            if self.verify_results(self.folder_path):
                path = self.main(index, file)
                et, eta = self.get_et_eta(path)
                self.save_results(self.folder_path,et, eta)
            if self.debug:
                break

        
    def get_et_eta(self, file_path):
        regex_pattern = r"et(\d+).*?eta(\d+)"
        match = re.search(regex_pattern, file_path)
        
        if match:
            et_number = match.group(1)
            eta_number = match.group(2)
            return et, eta


In [ ]:
class ModelV1(nn.Module):
    def __init__(self, input_dim):
        super(ModelV1, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Conv1d(in_channels=1, 
                               out_channels=4, 
                               kernel_size=2, 
                               padding='same') 
        self.conv2 = nn.Conv1d(in_channels=4, 
                               out_channels=8, 
                               kernel_size=2, 
                               padding='same')
        self.fc1_in_features = 8 * input_dim
        self.fc1 = nn.Linear(self.fc1_in_features, input_dim)
        self.fc2 = nn.Linear(input_dim, 1)

    def forward(self, x):
        x = x.view(-1, 1, self.input_dim) 
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

In [46]:
import re

file_path = "/photonRingerdasdasdasdsa/datasets/notIso/mc23_13TeV.sgn.gammajet.bkg.vetoMC.dijet_et0_eta0.npz"

def get_et_eta():
    regex_pattern = r"et(\d+).*?eta(\d+)"
    match = re.search(regex_pattern, file_path)
    
    if match:
        et_number = match.group(1)
        eta_number = match.group(2)
        return et, eta


0
0
